# ChargeGrid Intelligence — Sprint 4 · avaliação no Colab

Roda o projeto inteiro no Colab, sem instalar nada no computador:

1. clona o repositório e instala as dependências;
2. indexa a base de conhecimento no ChromaDB (`nomic-embed-text`);
3. roda as avaliações — a versão das Sprints 1/2, as 3 iterações do RAG, a comparação de
   modelos e os testes de segurança — com a rubrica de fallback aplicada pelo LLM-juiz;
4. preenche as tabelas dos relatórios e gera `docs/relatorio_evolucao.pdf`;
5. (opcional) sobe a interface web com link público.

**Antes de rodar:** no menu à esquerda, clique no ícone de chave (**Secrets**) e cadastre:

| Nome | Onde conseguir |
|---|---|
| `GROQ_API_KEY` | <https://console.groq.com/keys> |
| `OLLAMA_API_KEY` | <https://ollama.com> → Settings → Keys |
| `GOOGLE_API_KEY` *(recomendado)* | <https://aistudio.google.com/apikey> — grátis, usada se a cota diária da Groq acabar |
| `GITHUB_TOKEN` *(opcional)* | só para enviar os resultados de volta ao GitHub (seção 6) |

Ligue o acesso do notebook a cada segredo. Depois use **Ambiente de execução → Executar tudo**.
A rodada completa leva de 30 a 60 minutos por causa do limite da conta gratuita do Groq.

## 1. Projeto e dependências

In [ ]:
import os

REPO = "https://github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git"
PASTA = "/content/sprint-4-Jorge"

if not os.path.exists(PASTA):
    !git clone -q {REPO} {PASTA}
else:
    !git -C {PASTA} pull -q
%cd {PASTA}
!pip install -q -r requirements.txt

### Resultados salvos no Google Drive

Os resultados das avaliações ficam em `Meu Drive/sprint4_resultados/`, e não só dentro do
Colab. Se o Colab desconectar ou a cota acabar, na próxima vez o notebook continua de onde
parou. Na primeira vez, o Google pede para autorizar o acesso ao Drive.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/sprint4_resultados")
DRIVE.mkdir(parents=True, exist_ok=True)

local = Path(PASTA) / "evals" / "resultados"
if not local.is_symlink():
    for arquivo in local.glob("*.json"):          # o que já vem no repositório
        if not (DRIVE / arquivo.name).exists():
            shutil.copy(arquivo, DRIVE / arquivo.name)
    shutil.rmtree(local)
    local.symlink_to(DRIVE)
# o rodar_tudo copia o PDF para o Drive depois de cada etapa que termina
os.environ["COPIAR_PDF_PARA"] = str(DRIVE)
print("resultados já salvos no Drive:", sorted(a.name for a in DRIVE.glob("*.json")))

## 2. Chaves

In [ ]:
from google.colab import userdata

CHAVES = ("GROQ_API_KEY", "OLLAMA_API_KEY", "GOOGLE_API_KEY")
for nome in CHAVES:
    try:
        os.environ[nome] = userdata.get(nome) or ""
    except Exception:
        os.environ[nome] = ""
print("chaves carregadas:", [n for n in CHAVES if os.environ.get(n)])

## 3. Conferir as chaves e preparar os embeddings

Testa cada modelo de chat com uma pergunta curta e guarda só os que responderam (a
comparação de modelos da seção 5 usa essa lista). Depois testa o `nomic-embed-text` na
Ollama Cloud; se não funcionar, instala o Ollama dentro do Colab e usa o modelo local.

In [ ]:
from src.llm.provedores import GEMINI_CANDIDATOS, MODELOS_DISPONIVEIS, construir_llm

def responde(provedor, modelo):
    try:
        construir_llm(provedor, modelo, max_tokens=50).invoke("Responda só: ok")
        print(f"ok     {provedor}:{modelo}")
        return True
    except Exception as erro:
        print(f"FALHOU {provedor}:{modelo} -> {str(erro)[:120]}")
        return False

MODELOS_OK = [f"{p}:{m}" for p, m in MODELOS_DISPONIVEIS if p != "gemini" and responde(p, m)]
if os.environ.get("GOOGLE_API_KEY"):
    # o Google tira modelos antigos de contas novas: usa o primeiro nome que a chave aceitar
    for nome in GEMINI_CANDIDATOS:
        if responde("gemini", nome):
            MODELOS_OK.append(f"gemini:{nome}")
            break

if not MODELOS_OK:
    raise SystemExit("Nenhum modelo respondeu. Confira as chaves e rode de novo.")

# COMO USAR O GEMINI (se a chave dele funcionou acima):
#   "juiz"  -> a Groq responde e o Gemini só dá as notas. Aproveita o que já rodou
#              (nada é regerado; só as notas antigas são refeitas pelo Gemini),
#              mas precisa de cota da Groq para gerar as respostas que faltam.
#   "tudo"  -> o Gemini responde e julga. Não depende da Groq (termina hoje),
#              mas refaz o legado e as iterações com o Gemini (~220 chamadas).
# Os resultados já salvos no Drive (legado e iteração 1) foram feitos com "juiz":
# mantenha "juiz" para não refazer nada. Use "tudo" só se a Groq ficar dias sem cota.
USAR_GEMINI_PARA = "juiz"

GEMINI = next((m for m in MODELOS_OK if m.startswith("gemini:")), None)
SEM_GEMINI = ["groq:openai/gpt-oss-20b", "ollama:gemma4:cloud"]
if GEMINI and USAR_GEMINI_PARA == "tudo":
    GERACAO = GEMINI
else:
    GERACAO = next(m for m in SEM_GEMINI + MODELOS_OK if m in MODELOS_OK)
if GEMINI:
    JUIZ = GEMINI
elif "groq:openai/gpt-oss-120b" in MODELOS_OK:
    JUIZ = "groq:openai/gpt-oss-120b"
else:
    JUIZ = GERACAO
print("\nmodelos usados na comparação:", MODELOS_OK)
print("modelo que responde:", GERACAO, "· juiz:", JUIZ)

In [ ]:
import shutil, subprocess, time, urllib.request

def testar_embedding():
    from src.rag.embeddings import criar_embeddings
    return len(criar_embeddings().embed_query("teste de recarga"))

def sh(comando):
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError(f"falhou: {comando}")

OLLAMA_LOCAL = "http://127.0.0.1:11434"

def ollama_respondendo():
    try:
        urllib.request.urlopen(OLLAMA_LOCAL, timeout=3)
        return True
    except Exception:
        return False

def garantir_ollama():
    # sobe (ou religa) o servidor local do Ollama, se o notebook estiver usando ele
    if os.environ.get("OLLAMA_EMBED_BASE_URL") != OLLAMA_LOCAL or ollama_respondendo():
        return
    print("Servidor do Ollama fora do ar — religando...")
    binario = shutil.which("ollama") or "/usr/local/bin/ollama"
    subprocess.Popen(f"nohup {binario} serve > /content/ollama.log 2>&1 &", shell=True,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    for _ in range(60):
        if ollama_respondendo():
            return
        time.sleep(1)
    print(open("/content/ollama.log").read()[-2000:])
    raise RuntimeError("O Ollama local não subiu; veja o log acima.")

def subir_ollama_local():
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils")
    if not shutil.which("ollama"):
        sh("curl -fsSL https://ollama.com/install.sh | sh")
    os.environ["OLLAMA_EMBED_BASE_URL"] = OLLAMA_LOCAL
    garantir_ollama()
    sh(f"{shutil.which('ollama') or '/usr/local/bin/ollama'} pull nomic-embed-text")

try:
    print("Ollama Cloud ok, dimensão:", testar_embedding())
except Exception as erro:
    print("Ollama Cloud não serviu o nomic-embed-text:", str(erro)[:150])
    print("Instalando o Ollama no Colab (1 a 2 minutos)...")
    subir_ollama_local()
    print("Ollama local ok, dimensão:", testar_embedding())

## 4. Testes offline

In [ ]:
!python -m unittest discover -s tests

## 5. Avaliações — uma célula por etapa

Cada célula roda **uma** etapa, aplica a rubrica de fallback (`evals/fallback/rubrica_manual.md`)
com o LLM-juiz e, assim que termina, **atualiza o `relatorio_evolucao.pdf`** e copia para
`Meu Drive/sprint4_resultados/`. Os resultados ficam salvos no Drive: etapa que já terminou
não roda de novo.

Comece pela célula de **status**: ela mostra o que já foi feito e o que falta. Rode só as
etapas marcadas como `[falta]`, na ordem. Se a cota diária da Groq acabar, a célula para com
`[PAROU]` — continue no dia seguinte a partir dela.

**Ordem recomendada:** status → iteração 2 → iteração 3 → segurança → modelos. A Groq
gratuita libera 200 mil tokens por dia por modelo, e cada etapa gasta cerca de 50 mil do
`gpt-oss-20b`. Se aparecer `[PAROU] A cota DIARIA...`, nada daquela etapa foi gravado:
rode a mesma célula no dia seguinte. Antes de cada etapa, rode de novo as seções 1 a 3
(clonar, Drive, chaves, Ollama) se o Colab tiver reiniciado.

In [ ]:
# STATUS: o que já terminou e o que falta executar
ARGS = f'--geracao {GERACAO} --juiz {JUIZ} --modelos {" ".join(MODELOS_OK)}'
!python -m evals.rodar_tudo --status

### 5.1 Antes — chatbot das Sprints 1/2

In [ ]:
!python -m evals.rodar_tudo --so legado {ARGS}

### 5.2 Iteração 1 — chunk fixo 1000, prompt rag_v1 (linha de base)

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --so iter1 {ARGS}

### 5.3 Iteração 2 — chunk por seção 500, prompt rag_v2 (grounding + citação), limiar e recusa

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --so iter2 {ARGS}

### 5.4 Iteração 3 — prompt rag_v3 com exemplos + reescrita da pergunta encadeada

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --so iter3 {ARGS}

### 5.5 Comparação de modelos (os que responderam na seção 3)

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --so modelos {ARGS}

### 5.6 Testes de segurança (injeção via documento + bateria de ataques da Sprint 3)

In [ ]:
garantir_ollama()
!python -m evals.rodar_tudo --so seguranca {ARGS}

## 6. Relatórios e PDF

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("evals/historico_scores.md", encoding="utf-8").read()))

In [ ]:
# se a busca falhou com "Failed to connect to Ollama", veja o log do servidor:
# print(open("/content/ollama.log").read()[-3000:])
# o PDF também vai para o Drive
shutil.copy("docs/relatorio_evolucao.pdf", DRIVE / "relatorio_evolucao.pdf")
# baixa resultados, relatórios e o PDF
!zip -qr resultados_sprint4.zip evals/resultados evals/historico_scores.md docs prompts/rag/CHANGELOG.md
from google.colab import files
files.download("resultados_sprint4.zip")
files.download("docs/relatorio_evolucao.pdf")

### Enviar os resultados para o GitHub *(opcional)*

Precisa do segredo `GITHUB_TOKEN` (GitHub → Settings → Developer settings → Personal access
tokens, com permissão de escrita no repositório). Preencha nome e e-mail do autor dos commits.

In [ ]:
NOME_AUTOR = "Pedro Sampaio"
EMAIL_AUTOR = "pedrosampaiom2007@gmail.com"

token = userdata.get("GITHUB_TOKEN")
!git config user.name "{NOME_AUTOR}"
!git config user.email "{EMAIL_AUTOR}"
# evals/resultados aponta para o Drive (atalho); o git guardaria só o atalho, então
# copia os JSONs para uma pasta de verdade antes do commit e recria o atalho depois
local = Path(PASTA) / "evals" / "resultados"
if local.is_symlink():
    local.unlink()
    local.mkdir()
    for arquivo in DRIVE.glob("*.json"):
        shutil.copy(arquivo, local / arquivo.name)
!git add evals/resultados/*.json evals/historico_scores.md docs prompts/rag/CHANGELOG.md
!git commit -q -m "Resultados das avaliações da Sprint 4 (iterações, modelos e segurança)"
!git push -q https://{token}@github.com/pedrosampaiom2007-gif/sprint-4-Jorge.git HEAD:main
shutil.rmtree(local)
local.symlink_to(DRIVE)

## 7. Interface web *(opcional)*

In [ ]:
# gera um link público do Gradio (válido enquanto o notebook estiver rodando)
from app.web import construir_interface
construir_interface().launch(share=True)